# 07 -- Frequency content (Fourier, PSD, STFT)

In [1]:
%matplotlib inline

import numpy as np
from asdea_sensors import SensorDataset
from asdea_sensors.config import settings

# The example data folder (edit to your path if needed).
DATA = r"C:\Users\ppala\Desktop\02_31MAY2026"

# A short window keeps every example fast (one ~10 min file).
WSTART, WLEN = "2026-05-31 15:00:00", "10min"

In [2]:
ds = SensorDataset(DATA, verbose=True)
ds.devices

------------------------------------------------------------
SensorDataset
------------------------------------------------------------
path        : C:\Users\ppala\Desktop\02_31MAY2026
files       : 32
time span   : 2026-05-31 14:52:12  ->  2026-05-31 20:02:13
duration    : 18601.0 s
devices     : MNAT0031, MNAT0034, MOF00134, MOF00135, MOF00136
fs / dt     : 252.5885 Hz / 0.003959 s
------------------------------------------------------------
axes (per sensor):
  MNAT0031   -> (3, 1, 5)
  MNAT0034   -> (3, 1, 5)
  MOF00134   -> (0, 1, 2)
  MOF00135   -> (3, 1, 5)
  MOF00136   -> (3, 1, 5)
------------------------------------------------------------
on-disk size: 782.17 MB
RAM         : used 31.59 GB / avail 1.92 GB (94%)
------------------------------------------------------------


['MNAT0031', 'MNAT0034', 'MOF00134', 'MOF00135', 'MOF00136']

In [3]:
wf = ds.MOF00135.window(WSTART, WLEN).baseline().filter(0.1, 24.9)

## Fourier, raw and Konno-Ohmachi smoothed

In [4]:
print("raw peaks  :", [round(float(x),2) for x in wf.fourier(component="x", smooth=None)["dom_freqs"]])
print("konno peaks:", [round(float(x),2) for x in wf.fourier(component="x", smooth="konno", bexp=40)["dom_freqs"]])

C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MOF00135' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(


C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\derive\filters.py:49: UserWarning: obspy not available; falling back to the scipy engine
  warnings.warn("obspy not available; falling back to the scipy engine")


[fourier] MOF00135 comp=x nfreq=4 smooth=None (computed)
raw peaks  : [0.2, 14.51, 9.46, 17.77]


[fourier] MOF00135 comp=x nfreq=4 smooth=konno (computed)
konno peaks: [0.12]


## PSD and band energy

In [5]:
psd = wf.psd(component="x", nperseg=512, noverlap=256)
print({str(k): round(float(v),6) for k,v in psd["band_energy"].items()})

[psd] MOF00135 comp=x nperseg=512 (computed)
{'(0, 1)': 0.0, '(1, 2.5)': 0.0, '(2.5, 5)': 0.0, '(5, 10)': 0.0}


## STFT spectrogram

In [6]:
s = wf.stft(component="x", nperseg=256, noverlap=224, fmax=25.0)
print("Zxx shape:", s["Zxx"].shape, " f up to:", round(float(s["f"].max()),1), "Hz")

[stft] MOF00135 comp=x nperseg=256 fmax=25.0 (computed)
Zxx shape: (26, 4665)  f up to: 24.3 Hz
